### Create the CartPole environment

In [1]:
import gym
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


env = gym.make('CartPole-v0')

### Setup R packages

In [2]:
from cxrl.lib.r_to_py import setup_R
setup_R()

[]


### A policy (model-free) for cartpole example

In [ ]:
def policy(state):
    x0 = state[0]/2.4
    x2 = state[2]/2.095
    if x0*0.4+x2*0.6>=0:
        return 1
    else:
        return 0

### Infer the global causal graph from episodes

In [19]:
from cxrl.lib.xrl import XRL
# names for interpretable representations
feature_names = ['cart_position', 'cart_velocity', 'pole_angle', 'pole_velocity', 'push']
xrl = XRL(env, pi=policy, repr_names=feature_names, verbose=True, repr_integration=False, nepisodes=200)

experience size= 6224
 causes: ['cart_position', 'cart_velocity'] -> cart_position
 causes: ['cart_velocity', 'pole_velocity', 'push'] -> cart_velocity
 causes: ['pole_angle', 'pole_velocity'] -> pole_angle
 causes: ['pole_velocity', 'push'] -> pole_velocity
 causes: ['cart_position', 'pole_angle', 'push'] -> push
BART SLA elapsed time: 0 seconds


In [ ]:
states, actions, rewards, states_new = xrl.replay_buffer

from cxrl.lib.utils import convert_and_expand
states = convert_and_expand(states).astype(float)
actions = convert_and_expand(actions).astype(float)

### Evaluation

Extract timesteps where action is a target node in the local causal model

In [7]:
replay = pd.DataFrame(np.concatenate((states, actions), axis=1), columns=feature_names)
valid_steps = []
for step in range(replay.shape[0]):
    query = replay[step:step+1]
    localG = xrl.local_causal_network(query)
    count = 0
    for edge in localG.edges():
        if edge[1] == 'push' and edge[0] != 'push':
                    count += 1
        if count > 1:
            valid_steps.append(step)
            break
print("number of steps for evaluations:", len(valid_steps))

number of steps for evaluations: 5579


Create representation pools for timestep t and t+1

In [8]:
# remove the last valid timestep if it's replay's last step
if valid_steps[-1] == replay.shape[0]-1:
    valid_steps = valid_steps[:-1]
valid_steps_tp1 = (np.array(valid_steps)+1).tolist()
eval_replay_t = replay.iloc[valid_steps,:]
eval_replay_tp1 = replay.iloc[valid_steps_tp1,:]
print(eval_replay_t.head())
eval_replay_t = eval_replay_t.to_numpy()
eval_replay_tp1 = eval_replay_tp1.to_numpy()

   cart_position  cart_velocity  pole_angle  pole_velocity  push
0      -0.004850       0.013501   -0.038987       0.010646   0.0
1      -0.004580      -0.181041   -0.038774       0.290777   0.0
2      -0.008201      -0.375589   -0.032958       0.570984   0.0
3      -0.015713      -0.570234   -0.021538       0.853104   0.0
4      -0.027117      -0.765056   -0.004476       1.138937   0.0


Create holdout dataset

In [9]:
action_index = 4
features_t = eval_replay_t
action_tp1 = eval_replay_tp1[:,action_index]

X_train, X_test, y_train, y_test = train_test_split(features_t, action_tp1, test_size=0.4, random_state = 10)
train_idx, test_idx = train_test_split(np.arange(len(valid_steps)), test_size=0.4, random_state = 10)


Decision Tree accuracy

In [10]:
clf_tree = DecisionTreeClassifier(random_state=1)
clf_tree.fit(X_train, y_train)
y_pred_tree = clf_tree.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print(f'Decision Tree Accuracy: {accuracy_tree*100:.2f}%')

Decision Tree Accuracy: 94.09%


Logistic Regression accuracy

In [11]:
clf_lr = LogisticRegression(random_state=1).fit(X_train, y_train.flatten())
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr = accuracy_score(y_pred_lr, y_test.flatten())
print(f'Logistic Regression Accuracy: {accuracy_lr*100:.2f}%')


Logistic Regression Accuracy: 88.53%


Build BART inference model per treatment (parent nodes of action)

In [14]:
from cxrl.lib.models import BARTRegressor
global_cause = ['cart_position', 'pole_angle', 'push']
causes = []
covars = []
for i,treat in enumerate(feature_names):
        if treat in global_cause:
            causes.append(i)
        else:
             covars.append(i)
hash_y_bart = {}
for treat in causes:
    treat_and_covars = [treat] + covars
    bart = BARTRegressor()   
    bart.fit(X_train[:,treat_and_covars], y_train)
    hash_y_bart[treat] = bart.predict(X_test[:,treat_and_covars])

Predict action using the local causal model (weighted by edge strengths)

In [ ]:
action_index = 4
num_cols = [0, 1, 2, 3]
y_pred_bart = np.zeros(y_test.shape[0])
for i,step in enumerate(test_idx):
        query = replay[step:step+1]
        local_causes = xrl.scm[action_index].local_treatments(query)
        total_weight = 0
        y_Bart_pred = 0
        for treat in np.nonzero(local_causes[0])[0]:
                weight = abs(xrl.scm[action_index].ITE(query, int(treat), additive=(treat in num_cols)).mean())
                total_weight += weight
                y_Bart_pred += hash_y_bart[treat][i] * weight
        y_BART_pred = y_Bart_pred / total_weight
        y_pred_bart[i] = y_BART_pred

Our model accuracy

In [16]:
accuracy_bart = accuracy_score((y_pred_bart > 0.5).astype(np.int64), y_test.flatten())
print(f'BART Accuracy: {accuracy_bart*100:.2f}%')

BART Accuracy: 91.35%
